# 10 — RAG Project: Chat With Your Documents
### Capstone · Wiring together 06_Document_Loaders through 09_Retriever

Four notebooks built four separate pieces: loaders, splitters, a vector store, a retriever. This
notebook wires all four into one real application — plus the one piece none of them covered on
their own: **conversational memory**, so a follow-up question like *"how long does that take?"*
correctly resolves against whatever was asked two turns ago, instead of retrieving nonsense because
"that" means nothing on its own.

## The one new idea this project needs

A plain retriever (from `09_Retriever`) takes **one string** and searches for it. That's a problem
the moment a conversation has more than one turn:

> **Turn 1 — Human:** What's your refund policy?
> **Turn 2 — Human:** How long does that take?

Searching the vector store for the literal text `"How long does that take?"` retrieves nothing
useful — there's no vector in the store that's "about" the word "that." A real chatbot needs to
**rewrite "that" into "refund processing time"** using the conversation so far, *before* retrieving.

That's `create_history_aware_retriever`: an LLM looks at the chat history plus the new question,
produces a standalone, self-contained version of it, and only *then* does the retrieval step run.

> 📌 `create_history_aware_retriever`, `create_retrieval_chain`, and `create_stuff_documents_chain`
> all live in `langchain_classic.chains` now — same `langchain-classic` package introduced in
> `09_Retriever` for the retriever classes.

## Setup

In [ ]:
# %pip install -q langchain langchain-core langchain-text-splitters python-dotenv
# %pip install -q langchain-huggingface sentence-transformers langchain-chroma langchain-groq
# %pip install -q langchain-classic "langchain-community==0.4.2"


In [1]:
import os, shutil
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("Environment ready.")


Environment ready.


---
## Step 1 — Load (from `06_Document_Loaders`)

A small support knowledge base, built the same way as `06_Document_Loaders`'s capstone — this project
reuses that exact pattern instead of reinventing it.

In [2]:
DOCS_DIR = Path("sample_docs")
DOCS_DIR.mkdir(exist_ok=True)

(DOCS_DIR / "returns_policy.txt").write_text("""Returns Policy

Customers may return most items within 30 days of delivery, provided the item is unused and in its
original packaging. Refunds are issued to the original payment method within 5-7 business days after
we receive the returned item. Shipping fees are non-refundable unless the item arrived damaged or
defective. Digital products and gift cards cannot be returned once delivered.
""", encoding="utf-8")

(DOCS_DIR / "shipping_policy.txt").write_text("""Shipping Policy

Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days and is available
for orders placed before 2 PM. Orders over $50 qualify for free standard shipping. International
shipping is available to select countries and typically takes 7-14 business days.
""", encoding="utf-8")

(DOCS_DIR / "account_policy.txt").write_text("""Account & Security

To reset your password, choose 'Forgot password' on the sign-in page and follow the emailed link.
Two-factor authentication codes are sent by SMS and may take up to 2 minutes to arrive. You can
change your account email from Settings > Profile > Email Address at any time.
""", encoding="utf-8")

from langchain_community.document_loaders import DirectoryLoader, TextLoader

raw_documents = DirectoryLoader(
    str(DOCS_DIR), glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
).load()

print(f"{len(raw_documents)} documents loaded")


3 documents loaded


C:\Users\Deepak Kumar\AppData\Local\Temp\ipykernel_20952\3660824788.py:26: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


---
## Step 2 — Split (from `07_Text_Splitters`)

Recursive splitting at the sweet spot established in `07_Text_Splitters`'s research summary, with
real overlap so no idea gets cut cleanly in half at a chunk boundary.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_documents)

print(f"{len(raw_documents)} documents -> {len(chunks)} chunks")
print(chunks[0].metadata)


3 documents -> 5 chunks
{'source': 'sample_docs\\account_policy.txt'}


---
## Step 3 — Store (from `08_Vector_Store`)

Chroma, built fresh each run of this notebook for a clean demo — in a real deployment this would be
built once and reopened on every app start, exactly like `08_Vector_Store`'s persistence capstone.

In [4]:
CHROMA_PATH = "rag_project_kb"
shutil.rmtree(CHROMA_PATH, ignore_errors=True)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings,
    collection_name="rag_project", persist_directory=CHROMA_PATH,
)
print(f"{vectorstore._collection.count()} chunks indexed and persisted to '{CHROMA_PATH}/'")


d:\Gen Ai\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6970.43it/s]


5 chunks indexed and persisted to 'rag_project_kb/'


---
## Step 4 — Retrieve (from `09_Retriever`)

Plain similarity search, deliberately — this project is about assembling the pipeline correctly, not
about re-demonstrating MMR or hybrid search, which `09_Retriever` already covered. Swapping this one
line for `vectorstore.as_retriever(search_type="mmr", ...)` or an `EnsembleRetriever` drops straight
into everything below without changing anything else.

In [12]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("Base retriever and model ready.")


Base retriever and model ready.


---
## Step 5 — NEW: make the retriever history-aware

The missing piece. Given the chat history and a new question, an LLM call produces a standalone
version of the question — *then* that standalone version is what actually gets embedded and
searched. Notice the prompt is deliberately instructed to **only reformulate, never answer** — this
step's entire job is producing a better search query, nothing else.

In [13]:
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and the latest user question, rewrite the question as a "
               "standalone question that can be understood without the chat history. Do NOT answer "
               "the question — only reformulate it if needed, and return it as-is if it's already "
               "standalone."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, base_retriever, contextualize_prompt)
print("History-aware retriever ready.")


History-aware retriever ready.


---
## Step 6 — The answering chain

`create_stuff_documents_chain` is the "formally named" version of the `format_docs` + prompt pattern
from `06_Document_Loaders`'s capstone: it takes whatever documents the retriever found, "stuffs" them
into `{context}`, and generates the final answer.

In [14]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful support assistant. Answer using ONLY the context below. "
               "If the answer isn't in the context, say you don't have that information.\n\n"
               "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, answer_prompt)

# create_retrieval_chain wires them together: history-aware retrieval, THEN answering.
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)
print("Full RAG chain assembled: retrieve (history-aware) -> answer")


Full RAG chain assembled: retrieve (history-aware) -> answer


---
## Step 7 — Wrap with session memory

`RunnableWithMessageHistory` is what actually stores and replays `chat_history` between calls, keyed
by a `session_id` — so two different users (or two different conversations) never see each other's
history. The `InMemoryChatMessageHistory` store below is deliberately simple (a Python dict); a real
deployment would back this with Redis or a database so history survives a server restart.

In [15]:
session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)
print("Conversational RAG chain ready.")


Conversational RAG chain ready.


d:\Gen Ai\myvenv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


---
## The payoff: a real conversation, with a follow-up that only works because of Step 5

In [16]:
config = {"configurable": {"session_id": "demo-user-1"}}

response_1 = conversational_rag_chain.invoke({"input": "What's your refund policy?"}, config=config)
print("Q1:", "What's your refund policy?")
print("A1:", response_1["answer"])
print()

response_2 = conversational_rag_chain.invoke({"input": "How long does that take?"}, config=config)
print("Q2:", "How long does that take?")
print("A2:", response_2["answer"])


Q1: What's your refund policy?
A1: Our refunds are processed to the original payment method within 5‑7 business days after we receive your returned item. Shipping fees are not refundable unless the item arrived damaged or defective. Digital products and gift cards cannot be returned once delivered.

Q2: How long does that take?
A2: Refunds are issued to your original payment method within 5‑7 business days after we receive the returned item.


In [17]:
# See what the history-aware retriever hands back on turn 2. It first rewrites the vague follow-up
# using the history, THEN searches with the rewritten version — not the literal "How long does that take?"
rewritten = history_aware_retriever.invoke({
    "input": "How long does that take?",
    "chat_history": get_session_history("demo-user-1").messages[:2],
})
print("Chunks retrieved on turn 2, after the history-aware rewrite (refund-related ones should lead):")
for d in rewritten:
    print(" -", d.page_content[:70].replace("\n", " "))


Chunks retrieved on turn 2, after the history-aware rewrite (refund-related ones should lead):
 - Customers may return most items within 30 days of delivery, provided t
 - Shipping Policy  Standard shipping takes 3-5 business days. Express sh
 - Returns Policy


In [18]:
# A SECOND, independent session — proves sessions don't leak into each other.
response_other_user = conversational_rag_chain.invoke(
    {"input": "How long does that take?"},
    config={"configurable": {"session_id": "demo-user-2"}},
)
print("A brand-new session asked the SAME vague follow-up with NO prior turns:")
print(response_other_user["answer"][:200])
print("\n(With no history, there is nothing to resolve \"that\" against. A well-behaved assistant should say it",
      "needs more detail or lacks the information. Exact wording varies by model, so compare it with turn 2 above.)")


A brand-new session asked the SAME vague follow-up with NO prior turns:
I’m sorry, but I don’t have enough information to answer that question.

(With no history, there is nothing to resolve "that" against. A well-behaved assistant should say it needs more detail or lacks the information. Exact wording varies by model, so compare it with turn 2 above.)


---
## Inspecting what actually got retrieved — for debugging and for citations

`response["context"]` holds the actual `Document` objects the chain retrieved for that turn — useful
both for debugging ("why did it answer that?") and for showing users where an answer came from,
exactly like the `[source: ...]` citation pattern from `06_Document_Loaders`.

In [19]:
print("Documents retrieved for Q1:")
for doc in response_1["context"]:
    print(" -", doc.metadata.get("source", "unknown"), "->", doc.page_content[:60].replace("\n", " "))


Documents retrieved for Q1:
 - sample_docs\returns_policy.txt -> Returns Policy
 - sample_docs\returns_policy.txt -> Customers may return most items within 30 days of delivery, 
 - sample_docs\returns_policy.txt -> defective. Digital products and gift cards cannot be returne


---
## Running it as an actual terminal chat loop

Everything above, as the interactive script you'd actually run. This is the direct upgrade of the
`while True` chatbot loop you built back in `02_Prompts` — same shape, now backed by a real,
persistent, retrieval-grounded knowledge base instead of the model's own memorized training data.

In [22]:
def run_chat_loop(session_id: str = "cli-user"):
    print("Support bot ready. Type 'quit' to exit.\n")
    config = {"configurable": {"session_id": session_id}}
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in {"quit", "exit"}:
            print("Goodbye!")
            break
        result = conversational_rag_chain.invoke({"input": user_input}, config=config)
        print("Bot:", result["answer"], "\n")

# Uncomment to actually run it interactively:
run_chat_loop()


Support bot ready. Type 'quit' to exit.

Bot: Our refund policy is as follows:

- If you return an eligible item (unused and in its original packaging) within 30 days of delivery, we’ll process a refund to your original payment method.
- The refund is issued within 5‑7 business days after we receive the returned item.
- Shipping fees are not refundable, except when the item arrived damaged or defective.
- Digital products and gift cards cannot be returned or refunded once delivered. 

Bot: Refunds are processed within 5‑7 business days after we receive your returned item. 

Goodbye!


### Try it yourself
1. Uncomment and run `run_chat_loop()` — have a real multi-turn conversation, including at least one
   deliberately vague follow-up, and watch it resolve correctly.
2. Swap `base_retriever` for the `EnsembleRetriever` or `mmr` retriever from `09_Retriever/` — confirm
   everything downstream (history-awareness, answering, memory) keeps working unmodified.
3. Add a fourth policy document to `sample_docs/` covering a new topic, re-run Steps 1–3, and ask a
   question about it in Step 7's conversation — this is adding to a knowledge base live, the same
   pattern as `08_Vector_Store`'s incremental-add section.
4. Back `session_store` with a file (`json.dump`/`json.load` on each `InMemoryChatMessageHistory`'s
   messages) instead of a plain Python dict, so conversations survive a notebook restart.
5. Add a second department's documents with different `metadata["department"]` values, and apply a
   `filter` on `base_retriever`'s `search_kwargs` — confirm the history-aware step still resolves
   follow-ups correctly even with filtered retrieval underneath it.

---
**Next:** `11_Tools/` — this project answers questions from what it already knows. The next step is
giving the model the ability to *act*: call a function, look something up, take an action in the
world — starting with how to define a tool in the first place.
